# Chapter 6: Determinants

**Source span.** Perspectives on Projective Geometry, Chapter 6, sections 6.1-6.5, printed pages 93-108 / PDF pages 115-130.

**Chapter question.** How can determinants become the objects we calculate with, rather than expanded coordinate expressions we calculate through?

This notebook treats a determinant bracket as a geometric sensor. A bracket can record orientation, detect collinearity, construct joins and meets, and expose which polynomial statements survive projective transformations and homogeneous rescaling. The computations below are original examples built to match the chapter themes, not copied figures or source-page images.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
candidates = [START, *START.parents, START / "Perspectives-on-Projective-Geometry"]
for candidate in candidates:
    if (candidate / "AGENTS.md").exists() and (candidate / "artifacts").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Perspectives-on-Projective-Geometry course root.")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER = 6
CHAPTER_SLUG = "chapter-06-determinants"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
for child in ("figures", "html", "tables", "checks"):
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

def book_relative(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

book_relative(ARTIFACT_ROOT)


## Computational Translation Guide

The chapter moves from coordinate expansion to bracket algebra. In this notebook we use the following translation.

| Book language | Computational object | What we inspect |
| --- | --- | --- |
| Three homogeneous points are collinear | `det([p q r]) = [p,q,r]` | the determinant is zero exactly when the three lifted vectors are dependent |
| Orientation and affine area | `0.5 * [p,q,r]` after normalizing to `w=1` | sign changes under row/order swaps; magnitude is twice area |
| Join of two points and meet of two lines | cross products and determinant cofactors | incidence residuals such as `<line, point>` |
| Plucker's mu | a linear combination of two equations | the new object keeps shared zeros and is forced through one more point |
| Projective invariant property | zero set of a multihomogeneous bracket polynomial | all terms pick up the same nonzero scale factor |
| Grassmann-Plucker relation | a bracket polynomial that always vanishes | exact symbolic or integer checks plus an area decomposition view |

**Library routing.** Matplotlib is used for durable 2D determinant, area, incidence, and proof-state diagrams because this chapter is planar and sign-sensitive. Plotly is used for the scaling lab because comparing original, transformed, and predicted bracket values benefits from an interactive hoverable HTML artifact. SymPy is used where the chapter states algebraic identities that should vanish exactly, not merely numerically. NumPy handles homogeneous-coordinate arithmetic and deterministic numeric checks.


In [ ]:
import json
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_table
from utils.projective import affine, bracket, cross_ratio, hpoint, incidence, join, meet

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
})

COLORS = {
    "ink": "#222222",
    "blue": "#2f6f9f",
    "green": "#4f8f57",
    "red": "#b84a4a",
    "gold": "#c49a37",
    "purple": "#7c5ca3",
    "gray": "#808080",
}


def save_figure(fig, filename):
    path = ARTIFACT_ROOT / "figures" / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def det2(a, b):
    return float(np.linalg.det(np.column_stack([a, b])))


def signed_area2(a, b, c):
    return bracket(hpoint(*a), hpoint(*b), hpoint(*c))


def signed_area(a, b, c):
    return 0.5 * signed_area2(a, b, c)


def line_segment(line, xlim=(-2.2, 2.2), ylim=(-1.6, 1.7), tol=1e-10):
    a, b, c = [float(x) for x in line]
    points = []
    for x in xlim:
        if abs(b) > tol:
            y = -(a * x + c) / b
            if ylim[0] - 1e-8 <= y <= ylim[1] + 1e-8:
                points.append((x, y))
    for y in ylim:
        if abs(a) > tol:
            x = -(b * y + c) / a
            if xlim[0] - 1e-8 <= x <= xlim[1] + 1e-8:
                points.append((x, y))
    unique = []
    for p in points:
        if not any(np.linalg.norm(np.asarray(p) - np.asarray(q)) < 1e-7 for q in unique):
            unique.append(p)
    if len(unique) < 2:
        return None
    max_pair = None
    max_dist = -1.0
    for i in range(len(unique)):
        for j in range(i + 1, len(unique)):
            dist = np.linalg.norm(np.asarray(unique[i]) - np.asarray(unique[j]))
            if dist > max_dist:
                max_dist = dist
                max_pair = (unique[i], unique[j])
    return max_pair


def draw_line(ax, line, label, color, xlim=(-2.2, 2.2), ylim=(-1.6, 1.7), linestyle="-"):
    segment = line_segment(line, xlim=xlim, ylim=ylim)
    if segment is None:
        return
    (x0, y0), (x1, y1) = segment
    ax.plot([x0, x1], [y0, y1], color=color, lw=2.2, ls=linestyle, label=label)


def annotate_point(ax, point, label, color=COLORS["ink"], dx=0.04, dy=0.04):
    xy = affine(point)
    ax.scatter([xy[0]], [xy[1]], s=38, color=color, zorder=5)
    ax.text(xy[0] + dx, xy[1] + dy, label, color=color, weight="bold")


def clean_image_stats(path):
    data = image_stats(path)
    data["path"] = book_relative(path)
    return data


def bracket_polynomial_concurrency(points):
    a, b, c, d, e, f = [points[key] for key in "abcdef"]
    return bracket(c, d, b) * bracket(e, f, a) - bracket(c, d, a) * bracket(e, f, b)


def gp3_value(a, b, c, d, e, f):
    return (
        bracket(a, b, c) * bracket(d, e, f)
        - bracket(a, b, d) * bracket(c, e, f)
        + bracket(a, b, e) * bracket(c, d, f)
        - bracket(a, b, f) * bracket(c, d, e)
    )


## Visualization Planner Pass

The implementation route for this chapter is intentionally algebraic, but every algebraic statement gets an inspectable geometric view.

1. **Bracket orientation.** Show that `[a,b,c]` is the signed double area in an affine chart and that swapping order reverses orientation.
2. **Collinearity determinant test.** Move a point across a fixed line and watch the determinant pass through zero.
3. **Plucker mu.** Build a line through the intersection of two lines and a target point by combining two line equations.
4. **Concurrency bracket polynomial.** Test the condition that three joins meet at one point, then perturb one point to show the residual becomes nonzero.
5. **Invariant scaling.** Compare a bracket polynomial before and after `T * P * D`; every term receives the same nonzero multiplier.
6. **Grassmann-Plucker checks.** Verify the identities algebraically and draw the four-triangle oriented-area cancellation.

Artifacts are saved under `artifacts/chapter-06-determinants/` with concept names. The final cell asserts that every artifact exists, is nonempty, and matches the determinant identities used in the lesson.


In [ ]:
STORYBOARD = {
    "chapter goal": "Treat determinant brackets as first-class geometric objects for incidence, orientation, construction, and invariant tests.",
    "source span read": "Chapter 6, sections 6.1-6.5; printed pp. 93-108 / PDF pp. 115-130.",
    "concept inventory": [
        "bracket notation for 2x2 and 3x3 determinants",
        "collinearity and concurrence as determinant zero tests",
        "signed area and alternating orientation",
        "Plucker mu as a linear-combination construction",
        "multihomogeneous bracket polynomials and invariant scaling under T P D",
        "Grassmann-Plucker relations in 2x2 and 3x3 brackets",
    ],
    "library routing table": [
        {"concept": "signed area/orientation", "representation": "labeled planar determinant diagrams", "library": "Matplotlib", "why": "static equal-aspect area and orientation labels are the inspectable object"},
        {"concept": "Plucker mu and incidence", "representation": "line pencil diagram plus residual checks", "library": "NumPy + Matplotlib", "why": "cross products and dot residuals expose the construction directly"},
        {"concept": "invariant scaling", "representation": "interactive value comparison", "library": "Plotly", "why": "hoverable bars compare original, transformed, and predicted bracket polynomial values"},
        {"concept": "Grassmann-Plucker identities", "representation": "exact algebra checks", "library": "SymPy", "why": "the relation should simplify to exact zero"},
    ],
    "visual sequence": [
        {"artifact": "figures/bracket-signed-area-orientation.png", "inspection target": "orientation sign and double-area magnitude", "validation": "determinant equals twice signed area"},
        {"artifact": "figures/collinearity-determinant-test.png", "inspection target": "determinant zero at collinearity and sign change across the line", "validation": "zero residual for the on-line point"},
        {"artifact": "figures/plucker-mu-line-pencil.png", "inspection target": "the constructed line shares the old intersection and hits q", "validation": "two incidence residuals vanish"},
        {"artifact": "figures/concurrency-bracket-polynomial.png", "inspection target": "three joins concurrent exactly when the bracket polynomial vanishes", "validation": "residual zero before perturbation and nonzero after"},
        {"artifact": "html/invariant-scaling-bracket-lab.html", "inspection target": "common nonzero scale factor under T P D", "validation": "transformed value matches predicted factor times original"},
        {"artifact": "figures/grassmann-plucker-area-relation.png", "inspection target": "oriented triangle areas cancel", "validation": "2x2 symbolic and 3x3 exact integer checks vanish"},
    ],
    "risks": [
        "No textbook page images or copied figure layouts are used.",
        "Static diagrams are used where determinant signs are the main geometry; Plotly is reserved for the scaling comparison.",
    ],
}
storyboard_path = save_json(STORYBOARD, ARTIFACT_ROOT, "checks", "storyboard.json")
book_relative(storyboard_path)


## 1. Brackets Carry Orientation

For affine representatives `a=(x_a,y_a,1)`, `b=(x_b,y_b,1)`, and `c=(x_c,y_c,1)`, the bracket `[a,b,c]` is twice the signed area of the triangle. This is the first reason determinants are useful as geometric atoms: the same scalar records area magnitude, orientation, and degeneracy.

Inspect the figure for two signs and one zero case. The bracket changes sign when two entries are swapped and collapses to zero when the three points lie on a line.


In [ ]:
orientation_cases = [
    ("positive order", np.array([0.0, 0.0]), np.array([1.5, 0.25]), np.array([0.35, 1.15]), COLORS["green"]),
    ("swapped order", np.array([0.0, 0.0]), np.array([0.35, 1.15]), np.array([1.5, 0.25]), COLORS["red"]),
    ("collinear", np.array([0.0, 0.0]), np.array([0.75, 0.38]), np.array([1.5, 0.75]), COLORS["gray"]),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
orientation_checks = []
for ax, (title, a2, b2, c2, color) in zip(axes, orientation_cases):
    det = signed_area2(a2, b2, c2)
    area = signed_area(a2, b2, c2)
    poly = Polygon([a2, b2, c2], closed=True, facecolor=color, alpha=0.18, edgecolor=color, linewidth=2)
    ax.add_patch(poly)
    ax.plot([a2[0], b2[0], c2[0], a2[0]], [a2[1], b2[1], c2[1], a2[1]], color=color, lw=2)
    for xy, label in [(a2, "a"), (b2, "b"), (c2, "c")]:
        ax.scatter([xy[0]], [xy[1]], s=42, color=color)
        ax.text(xy[0] + 0.04, xy[1] + 0.04, label, weight="bold")
    ax.set_title(title)
    ax.text(0.02, -0.18, f"[a,b,c] = {det:.3f}\narea = {area:.3f}", transform=ax.transAxes)
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, 1.8)
    ax.set_ylim(-0.2, 1.35)
    ax.grid(True, alpha=0.25)
    orientation_checks.append({"case": title, "bracket": det, "twice_area": 2 * area, "matches": abs(det - 2 * area) < 1e-12})

orientation_path = save_figure(fig, "bracket-signed-area-orientation.png")
save_json({"checks": orientation_checks}, ARTIFACT_ROOT, "checks", "bracket-orientation-checks.json")
display_artifact(orientation_path, width=900)


## 2. Collinearity Is a Determinant Test

The expanded coordinate formula for collinearity is not the conceptual object. The bracket is. If `a`, `b`, and `x` are homogeneous point vectors, `[a,b,x]=0` says exactly that `x` lies on the join `a v b`.

The next artifact fixes a line through `a` and `b`, then moves `x` across it. The determinant sweep is a signed distance proxy: it is zero on the line and changes sign on opposite sides.


In [ ]:
a2 = np.array([-1.45, -0.55])
b2 = np.array([1.45, 0.62])
base = b2 - a2
unit_normal = np.array([-base[1], base[0]]) / np.linalg.norm(base)
midpoint = 0.54 * a2 + 0.46 * b2
offsets = np.linspace(-0.75, 0.75, 17)
collinearity_rows = []
for offset in offsets:
    x2 = midpoint + offset * unit_normal
    det = signed_area2(a2, b2, x2)
    collinearity_rows.append({"offset": float(offset), "bracket_a_b_x": float(det)})
collinearity_table_path = save_table(collinearity_rows, ARTIFACT_ROOT, "tables", "collinearity-determinant-sweep.csv")

x_on = hpoint(*midpoint)
x_plus = hpoint(*(midpoint + 0.55 * unit_normal))
x_minus = hpoint(*(midpoint - 0.55 * unit_normal))
line_ab = join(hpoint(*a2), hpoint(*b2))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
draw_line(ax0, line_ab, "a v b", COLORS["blue"], xlim=(-1.8, 1.8), ylim=(-1.15, 1.25))
for point, label, color in [(hpoint(*a2), "a", COLORS["blue"]), (hpoint(*b2), "b", COLORS["blue"]), (x_on, "x0", COLORS["green"]), (x_plus, "x+", COLORS["red"]), (x_minus, "x-", COLORS["purple"])]:
    annotate_point(ax0, point, label, color=color)
for p0, p1, color in [(midpoint, midpoint + 0.55 * unit_normal, COLORS["red"]), (midpoint, midpoint - 0.55 * unit_normal, COLORS["purple"])]:
    ax0.arrow(p0[0], p0[1], p1[0] - p0[0], p1[1] - p0[1], width=0.01, head_width=0.07, color=color, length_includes_head=True, alpha=0.75)
ax0.set_title("point crosses the join a v b")
ax0.set_aspect("equal")
ax0.set_xlim(-1.8, 1.8)
ax0.set_ylim(-1.15, 1.25)
ax0.grid(True, alpha=0.25)
ax0.legend(loc="lower right")

ax1.axhline(0, color=COLORS["ink"], lw=1)
ax1.axvline(0, color=COLORS["gray"], lw=1, ls="--")
ax1.plot([row["offset"] for row in collinearity_rows], [row["bracket_a_b_x"] for row in collinearity_rows], marker="o", color=COLORS["gold"])
ax1.set_title("[a,b,x] as x crosses the line")
ax1.set_xlabel("signed offset from the line")
ax1.set_ylabel("determinant bracket")
ax1.grid(True, alpha=0.25)

collinearity_path = save_figure(fig, "collinearity-determinant-test.png")
collinearity_checks = {
    "line_incidence_a": float(np.dot(line_ab, hpoint(*a2))),
    "line_incidence_b": float(np.dot(line_ab, hpoint(*b2))),
    "on_line_bracket": float(bracket(hpoint(*a2), hpoint(*b2), x_on)),
    "positive_side_bracket": float(bracket(hpoint(*a2), hpoint(*b2), x_plus)),
    "negative_side_bracket": float(bracket(hpoint(*a2), hpoint(*b2), x_minus)),
    "table": book_relative(collinearity_table_path),
}
save_json(collinearity_checks, ARTIFACT_ROOT, "checks", "collinearity-determinant-checks.json")
display_artifact(collinearity_path, width=860)


## 3. Plucker's Mu as a Construction Device

Plucker's mu says that if two equations vanish on a common base set, then every linear combination of them also vanishes there. To force the combination through a new point `q`, use the coefficients obtained by evaluating the old equations at `q`.

For two lines `l1` and `l2`, the line

`<l2,q> l1 - <l1,q> l2`

passes through the old intersection `l1 meet l2` and through `q`. The figure makes the pencil visible, while the checks record the two incidence residuals.


In [ ]:
p1 = hpoint(-1.6, -0.82)
p2 = hpoint(1.25, 0.58)
p3 = hpoint(-1.35, 1.12)
p4 = hpoint(1.55, -0.88)
q = hpoint(0.72, 1.18)
l1 = join(p1, p2)
l2 = join(p3, p4)
base_point = meet(l1, l2)
mu_line = float(np.dot(l2, q)) * l1 - float(np.dot(l1, q)) * l2

fig, ax = plt.subplots(figsize=(7.2, 5.2), constrained_layout=True)
draw_line(ax, l1, "l1", COLORS["blue"], xlim=(-1.95, 1.95), ylim=(-1.25, 1.35))
draw_line(ax, l2, "l2", COLORS["purple"], xlim=(-1.95, 1.95), ylim=(-1.25, 1.35))
draw_line(ax, mu_line, "mu line", COLORS["red"], xlim=(-1.95, 1.95), ylim=(-1.25, 1.35))
for point, label, color in [(p1, "p1", COLORS["blue"]), (p2, "p2", COLORS["blue"]), (p3, "p3", COLORS["purple"]), (p4, "p4", COLORS["purple"]), (base_point, "l1 meet l2", COLORS["green"]), (q, "q", COLORS["red"])]:
    annotate_point(ax, point, label, color=color, dx=0.04, dy=0.04)
ax.set_title("Plucker mu: preserve the old intersection and force q")
ax.set_aspect("equal")
ax.set_xlim(-1.95, 1.95)
ax.set_ylim(-1.25, 1.35)
ax.grid(True, alpha=0.25)
ax.legend(loc="lower left")

plucker_path = save_figure(fig, "plucker-mu-line-pencil.png")
plucker_checks = {
    "mu_incidence_q": float(np.dot(mu_line, q)),
    "mu_incidence_intersection": float(np.dot(mu_line, base_point)),
    "l1_incidence_intersection": float(np.dot(l1, base_point)),
    "l2_incidence_intersection": float(np.dot(l2, base_point)),
    "mu_coeff_l1": float(np.dot(l2, q)),
    "mu_coeff_l2": float(-np.dot(l1, q)),
}
save_json(plucker_checks, ARTIFACT_ROOT, "checks", "plucker-mu-line-pencil-checks.json")
display_artifact(plucker_path, width=760)


## 4. Concurrency as a Bracket Polynomial

For three joins `a v b`, `c v d`, and `e v f`, the chapter derives a bracket condition of the form

`[c,d,b][e,f,a] - [c,d,a][e,f,b] = 0`.

This expression is not a coordinate coincidence. It is the incidence test obtained by first writing the meet of `a v b` and `c v d` with Plucker's mu, then testing whether that point lies on `e v f`. The artifact below shows a concurrent configuration and a one-point perturbation that breaks the condition.


In [ ]:
origin = np.array([0.12, 0.05])
directions = [np.array([1.25, 0.45]), np.array([-0.32, 1.05]), np.array([0.78, -0.94])]
labels = "abcdef"
coords = {
    "a": origin + 0.95 * directions[0],
    "b": origin - 0.85 * directions[0],
    "c": origin + 0.88 * directions[1],
    "d": origin - 0.78 * directions[1],
    "e": origin + 0.92 * directions[2],
    "f": origin - 0.72 * directions[2],
}
points = {key: hpoint(*value) for key, value in coords.items()}
perturbed_points = dict(points)
perturbed_points["f"] = hpoint(*(coords["f"] + np.array([0.32, 0.18])))
concurrency_value = bracket_polynomial_concurrency(points)
perturbed_value = bracket_polynomial_concurrency(perturbed_points)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for pair, color in [("ab", COLORS["blue"]), ("cd", COLORS["purple"]), ("ef", COLORS["green"])]:
    line = join(points[pair[0]], points[pair[1]])
    draw_line(ax0, line, f"{pair[0]} v {pair[1]}", color, xlim=(-1.35, 1.55), ylim=(-1.05, 1.35))
for key in labels:
    annotate_point(ax0, points[key], key, color=COLORS["ink"])
ax0.scatter([origin[0]], [origin[1]], s=70, marker="x", color=COLORS["red"], zorder=6)
ax0.text(origin[0] + 0.05, origin[1] + 0.05, "common meet", color=COLORS["red"], weight="bold")
ax0.set_title("three joins share one point")
ax0.set_aspect("equal")
ax0.set_xlim(-1.35, 1.55)
ax0.set_ylim(-1.05, 1.35)
ax0.grid(True, alpha=0.25)
ax0.legend(loc="upper left", fontsize=8)

bars = ["concurrent", "after moving f"]
values = [concurrency_value, perturbed_value]
bar_colors = [COLORS["green"], COLORS["red"]]
ax1.axhline(0, color=COLORS["ink"], lw=1)
ax1.bar(bars, values, color=bar_colors, alpha=0.82)
ax1.set_title("bracket polynomial residual")
ax1.set_ylabel("[c,d,b][e,f,a] - [c,d,a][e,f,b]")
for idx, value in enumerate(values):
    ax1.text(idx, value + (0.02 if value >= 0 else -0.02), f"{value:.3g}", ha="center", va="bottom" if value >= 0 else "top")
ax1.grid(True, axis="y", alpha=0.25)

concurrency_path = save_figure(fig, "concurrency-bracket-polynomial.png")
concurrency_checks = {
    "concurrent_residual": float(concurrency_value),
    "perturbed_residual": float(perturbed_value),
    "perturbed_is_nonzero": bool(abs(perturbed_value) > 1e-6),
}
save_json(concurrency_checks, ARTIFACT_ROOT, "checks", "concurrency-bracket-polynomial-checks.json")
display_artifact(concurrency_path, width=880)


## 5. Why Multihomogeneous Bracket Polynomials Are Invariant

A projective transformation sends each 3-bracket to `det(T)` times the original bracket. Rescaling individual homogeneous representatives sends a bracket to the product of the relevant point scales. A multihomogeneous bracket polynomial is built so every summand receives the same multiplier.

The lab below uses the concurrency polynomial `Q`. The transformed value is compared against the predicted factor

`det(T)^2 * lambda_a * lambda_b * lambda_c * lambda_d * lambda_e * lambda_f`.

Because this factor is nonzero, `Q=0` and `Q!=0` are both preserved.


In [ ]:
generic_points = {
    "a": hpoint(-1.2, 0.15),
    "b": hpoint(0.95, -0.58),
    "c": hpoint(-0.42, 1.05),
    "d": hpoint(1.28, 0.76),
    "e": hpoint(-1.05, -0.92),
    "f": hpoint(0.55, 1.36),
}
T = np.array([[1.25, 0.28, -0.18], [-0.22, 0.94, 0.31], [0.08, -0.16, 1.12]], dtype=float)
lambdas = {"a": 1.4, "b": -0.75, "c": 1.15, "d": 0.62, "e": -1.35, "f": 0.9}
transformed_points = {key: lambdas[key] * (T @ value) for key, value in generic_points.items()}
q_original = bracket_polynomial_concurrency(generic_points)
q_transformed = bracket_polynomial_concurrency(transformed_points)
common_factor = (float(np.linalg.det(T)) ** 2) * math.prod(lambdas.values())
q_predicted = common_factor * q_original
scaling_residual = q_transformed - q_predicted

term_rows = []
for label, point_set in [("original", generic_points), ("transformed", transformed_points)]:
    a, b, c, d, e, f = [point_set[key] for key in "abcdef"]
    term1 = bracket(c, d, b) * bracket(e, f, a)
    term2 = bracket(c, d, a) * bracket(e, f, b)
    term_rows.append({"state": label, "term_[c,d,b][e,f,a]": float(term1), "term_[c,d,a][e,f,b]": float(term2), "Q": float(term1 - term2)})
term_rows.append({"state": "predicted transformed", "term_[c,d,b][e,f,a]": None, "term_[c,d,a][e,f,b]": None, "Q": float(q_predicted)})
term_rows.append({"state": "common multiplier det(T)^2 product(lambda_i)", "term_[c,d,b][e,f,a]": None, "term_[c,d,a][e,f,b]": None, "Q": float(common_factor)})
for key in "abcdef":
    term_rows.append({"state": f"homogeneous scale lambda_{key}", "term_[c,d,b][e,f,a]": None, "term_[c,d,a][e,f,b]": None, "Q": float(lambdas[key])})
scaling_table_path = save_table(term_rows, ARTIFACT_ROOT, "tables", "bracket-invariant-scaling-table.csv")

fig = go.Figure()
fig.add_bar(x=["Q original", "Q transformed", "factor * Q original"], y=[q_original, q_transformed, q_predicted], marker_color=[COLORS["blue"], COLORS["red"], COLORS["green"]])
fig.update_layout(
    title="Invariant scaling of a multihomogeneous bracket polynomial",
    yaxis_title="polynomial value",
    template="plotly_white",
    width=820,
    height=460,
    annotations=[dict(text=f"common factor = {common_factor:.6g}<br>residual = {scaling_residual:.3e}", x=0.5, y=1.08, xref="paper", yref="paper", showarrow=False)],
)
scaling_html_path = ARTIFACT_ROOT / "html" / "invariant-scaling-bracket-lab.html"
fig.write_html(scaling_html_path, include_plotlyjs="cdn", full_html=True)

scaling_checks = {
    "det_T": float(np.linalg.det(T)),
    "lambda_product": float(math.prod(lambdas.values())),
    "common_factor": float(common_factor),
    "q_original": float(q_original),
    "q_transformed": float(q_transformed),
    "q_predicted": float(q_predicted),
    "scaling_residual": float(scaling_residual),
    "table": book_relative(scaling_table_path),
}
save_json(scaling_checks, ARTIFACT_ROOT, "checks", "invariant-scaling-bracket-checks.json")
display_artifact(scaling_html_path, width="100%", height=500)


## 6. Grassmann-Plucker Relations

The chapter ends by identifying bracket polynomials that vanish for every placement of the points. These are not geometric special cases; they are algebraic relations among determinants.

For four points on a projective line, the identity is

`[a,b][c,d] - [a,c][b,d] + [a,d][b,c] = 0`.

For six vectors in `R^3`, the projective-plane relation is

`[a,b,c][d,e,f] - [a,b,d][c,e,f] + [a,b,e][c,d,f] - [a,b,f][c,d,e] = 0`.

The figure draws the area interpretation of the four-term relation after an affine normalization: four oriented triangles cancel.


In [ ]:
c2 = np.array([-1.15, 0.05])
d2 = np.array([-0.12, 1.08])
e2 = np.array([1.25, 0.38])
f2 = np.array([0.42, -0.82])
triangle_specs = [
    ("+ Delta(d,e,f)", d2, e2, f2, 1, COLORS["green"]),
    ("- Delta(c,e,f)", c2, e2, f2, -1, COLORS["red"]),
    ("+ Delta(c,d,f)", c2, d2, f2, 1, COLORS["green"]),
    ("- Delta(c,d,e)", c2, d2, e2, -1, COLORS["red"]),
]
area_terms = []
fig, axes = plt.subplots(1, 4, figsize=(13, 3.8), constrained_layout=True)
for ax, (title, p, q_, r, sign, color) in zip(axes, triangle_specs):
    area = signed_area(p, q_, r)
    area_terms.append(sign * area)
    ax.add_patch(Polygon([p, q_, r], closed=True, facecolor=color, alpha=0.18, edgecolor=color, lw=2))
    for xy, label in [(c2, "c"), (d2, "d"), (e2, "e"), (f2, "f")]:
        ax.scatter([xy[0]], [xy[1]], s=32, color=COLORS["ink"])
        ax.text(xy[0] + 0.035, xy[1] + 0.035, label, weight="bold")
    ax.set_title(title)
    ax.text(0.02, -0.18, f"signed term = {sign * area:.3f}", transform=ax.transAxes)
    ax.set_aspect("equal")
    ax.set_xlim(-1.35, 1.45)
    ax.set_ylim(-1.0, 1.25)
    ax.grid(True, alpha=0.22)
fig.suptitle(f"Grassmann-Plucker area cancellation: sum = {sum(area_terms):.3e}", y=1.05)

gp_area_path = save_figure(fig, "grassmann-plucker-area-relation.png")

a1, a2, b1, b2, c1, c2s, d1, d2s = sp.symbols("a1 a2 b1 b2 c1 c2 d1 d2")
def det2s(u, v):
    return sp.Matrix([[u[0], v[0]], [u[1], v[1]]]).det()
avec, bvec, cvec, dvec = (a1, a2), (b1, b2), (c1, c2s), (d1, d2s)
gp2_expr = sp.expand(det2s(avec, bvec) * det2s(cvec, dvec) - det2s(avec, cvec) * det2s(bvec, dvec) + det2s(avec, dvec) * det2s(bvec, cvec))

rng = np.random.default_rng(202606)
integer_gp3_values = []
common_x_values = []
for _ in range(20):
    vecs = [rng.integers(-4, 5, size=3).astype(float) for _ in range(6)]
    integer_gp3_values.append(float(gp3_value(*vecs)))
    x, a, b, c, d = [rng.integers(-4, 5, size=3).astype(float) for _ in range(5)]
    common_x = bracket(x, a, b) * bracket(x, c, d) - bracket(x, a, c) * bracket(x, b, d) + bracket(x, a, d) * bracket(x, b, c)
    common_x_values.append(float(common_x))

gp_checks = {
    "gp2_symbolic_zero": bool(gp2_expr == 0),
    "gp3_max_abs_integer_residual": float(max(abs(v) for v in integer_gp3_values)),
    "common_x_max_abs_residual": float(max(abs(v) for v in common_x_values)),
    "area_relation_sum": float(sum(area_terms)),
}
save_json(gp_checks, ARTIFACT_ROOT, "checks", "grassmann-plucker-relation-checks.json")
display_artifact(gp_area_path, width=980)


## Applied Lab: Diagnose a Bracket Statement

Use this small report as a template for testing a proposed projective statement.

1. Write the statement as a bracket polynomial `Q`.
2. Check whether every term has the same number of brackets.
3. Check whether each point label occurs equally often in every term.
4. Apply a random projective transformation and random homogeneous rescalings.
5. Compare the transformed value with the predicted common factor.

The point of the lab is not to expand the determinant. It is to audit the bracket expression as a geometric object.


In [ ]:
def monomial_degree(labels_in_brackets):
    counts = {key: 0 for key in "abcdef"}
    for bracket_labels in labels_in_brackets:
        for key in bracket_labels:
            counts[key] += 1
    return counts

lab_terms = [
    {"name": "[c,d,b][e,f,a]", "brackets": ["cdb", "efa"]},
    {"name": "[c,d,a][e,f,b]", "brackets": ["cda", "efb"]},
]
lab_report = []
for term in lab_terms:
    lab_report.append({
        "term": term["name"],
        "number_of_brackets": len(term["brackets"]),
        "point_degrees": monomial_degree(term["brackets"]),
    })

same_bracket_count = len({row["number_of_brackets"] for row in lab_report}) == 1
same_point_degrees = all(lab_report[0]["point_degrees"] == row["point_degrees"] for row in lab_report[1:])
{
    "lab_report": lab_report,
    "multihomogeneous": bool(same_bracket_count and same_point_degrees),
    "scaling_residual_from_previous_lab": scaling_residual,
}


## Artifact Gallery

The main concept artifacts are linked here so the notebook remains readable before execution.

![Bracket signed area and orientation](../../artifacts/chapter-06-determinants/figures/bracket-signed-area-orientation.png)

![Collinearity determinant test](../../artifacts/chapter-06-determinants/figures/collinearity-determinant-test.png)

![Plucker mu line pencil](../../artifacts/chapter-06-determinants/figures/plucker-mu-line-pencil.png)

![Concurrency bracket polynomial](../../artifacts/chapter-06-determinants/figures/concurrency-bracket-polynomial.png)

![Grassmann-Plucker area relation](../../artifacts/chapter-06-determinants/figures/grassmann-plucker-area-relation.png)

Interactive lab: [invariant scaling bracket lab](../../artifacts/chapter-06-determinants/html/invariant-scaling-bracket-lab.html).


## Final Sanity Checks

The final checks intentionally mix geometry, algebra, and artifact integrity. A determinant notebook is only successful if the diagrams exist and the identities used to justify them actually vanish within the promised tolerances.


In [ ]:
figure_paths = [orientation_path, collinearity_path, plucker_path, concurrency_path, gp_area_path]
artifact_paths = figure_paths + [scaling_html_path, collinearity_table_path, scaling_table_path, storyboard_path]
assert_artifacts(artifact_paths)

assert all(item["matches"] for item in orientation_checks)
assert abs(collinearity_checks["on_line_bracket"]) < 1e-10
assert collinearity_checks["positive_side_bracket"] * collinearity_checks["negative_side_bracket"] < 0
assert abs(plucker_checks["mu_incidence_q"]) < 1e-10
assert abs(plucker_checks["mu_incidence_intersection"]) < 1e-10
assert abs(concurrency_checks["concurrent_residual"]) < 1e-10
assert concurrency_checks["perturbed_is_nonzero"]
assert abs(scaling_checks["scaling_residual"]) < 1e-10
assert gp_checks["gp2_symbolic_zero"]
assert gp_checks["gp3_max_abs_integer_residual"] < 1e-9
assert gp_checks["common_x_max_abs_residual"] < 1e-9
assert abs(gp_checks["area_relation_sum"]) < 1e-12

sample = [-1.4, -0.2, 0.75, 1.6]
image = [(1.1 * x - 0.25) / (0.22 * x + 1.0) for x in sample]
cross_ratio_error = abs(cross_ratio(*sample) - cross_ratio(*image))
assert cross_ratio_error < 1e-12

raster_stats = [clean_image_stats(path) for path in figure_paths]
assert all(item["file_size"] > 1000 and item["pixel_std"] > 5 for item in raster_stats)
all_files_exist = all(Path(path).exists() and Path(path).stat().st_size > 256 for path in artifact_paths)

visual_checks = {
    "chapter": CHAPTER,
    "all_files_exist": all_files_exist,
    "visual_count": len(artifact_paths),
    "raster_artifacts": raster_stats,
    "html_artifact": book_relative(scaling_html_path),
    "table_artifacts": [book_relative(collinearity_table_path), book_relative(scaling_table_path)],
    "cross_ratio_error": float(cross_ratio_error),
    "numeric_checks": {
        "collinearity_zero_bracket": float(collinearity_checks["on_line_bracket"]),
        "plucker_mu_q_incidence": float(plucker_checks["mu_incidence_q"]),
        "plucker_mu_base_incidence": float(plucker_checks["mu_incidence_intersection"]),
        "concurrency_residual": float(concurrency_checks["concurrent_residual"]),
        "invariant_scaling_residual": float(scaling_checks["scaling_residual"]),
        "grassmann_plucker_area_sum": float(gp_checks["area_relation_sum"]),
        "complex_cross_ratio_residual": float(cross_ratio_error),
    },
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    "chapter": CHAPTER,
    "source_span": "printed pp. 93-108 / PDF pp. 115-130",
    "artifacts": [book_relative(path) for path in artifact_paths],
    "checks": visual_checks,
    "identity_checks": {
        "orientation_determinant_equals_twice_area": True,
        "collinearity_bracket_zero": True,
        "plucker_mu_incidence": True,
        "concurrency_bracket_polynomial_zero": True,
        "invariant_scaling_factor": True,
        "grassmann_plucker_relations": True,
    },
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
{"visual_checks": book_relative(visual_checks_path), "final_sanity": book_relative(final_sanity_path)}


## Takeaways

- Brackets let us name determinant expressions at the right geometric level. Instead of expanding collinearity into coordinates, we use `[a,b,c]=0` directly.
- In an affine chart, the same bracket gives signed double area, so determinant sign is an orientation measurement.
- Plucker's mu is a construction rule: combine two equations so their shared zeros remain and a new constraint is forced.
- Multihomogeneous bracket polynomials define projectively invariant zero sets because every term acquires the same nonzero factor under `T * P * D`.
- Grassmann-Plucker relations are universal cancellations among brackets. They explain why different-looking bracket formulas can encode the same projective incidence property.
